# Step-projected three-step GC area comparison

This experiment evaluates the BM4-based method that projects two uncoupled guiding-center copies onto their mean after every direct or adjoint map. A complete BM4 step therefore contains twelve projections and uses no harmonic numerical coupling.

The study uses one 16-point circular boundary and three step sizes: $\pi/40$, $\pi/80$, and $\pi/160$. States and projected diagnostics are synchronized every $\pi/8$ up to $4\pi$.

The final animation combines the transported contours, relative area error, projected symplectic defect, and relative separation of the two internal GC copies.

API migration: this notebook uses the current simulation API. Physical rho and eta belong to dynamics or study settings; initial configurations store geometry. Stored outputs were cleared and should be regenerated before scientific interpretation. The former per-stage projection has been replaced explicitly by BM4Implicit with one projection around the complete cycle; this is a new method comparison.


In [ ]:
import numpy as np

from studies import (
    AreaComparisonConfig,
    RandomPotentialConfig,
    centered_circle,
    pi_area_steps,
    run_area_comparison,
)
from visualization import display_animation

In [ ]:
potential_config = RandomPotentialConfig(
    amplitude=0.7,
    max_wave_number=25,
    nx=64,
    ny=64,
    seed=27,
    interpolation_order=5,
)
potential = potential_config.build()
circle_radius = 0.5
circle_points = 16
rho = 0.3
circle = centered_circle(
    potential,
    radius=circle_radius,
    points=circle_points,
    
)
comparison_config = AreaComparisonConfig(
    rho=rho,
    steps=pi_area_steps(40, 80, 160),
    t_span=(0.0, 4 * np.pi),
    save_interval=np.pi / 8,
    coupling_frequency=0.0,
    progress=True,
)

print(potential_config)
print(
    f"Circle: {circle_points} points; t={comparison_config.t_span}; "
    f"{comparison_config.output_sample_count} saved states"
)

## Integrations and projected observations

The study derives the output grid, diagnostic stride, observer lifecycle, metadata blocks, and result mappings from the configuration above.

In [ ]:
result = run_area_comparison(
    potential,
    circle,
    notebook_path=(
        "notebooks/experiments/symplecticity/"
        "gc_area_and_projected_symplecticity_3.ipynb"
    ),
    config=comparison_config,
    metadata={
        **potential_config.metadata(),
        "circle_radius": circle_radius,
    },
)
result.print_summary()

## Comparative animation

The same color identifies each integration step in all four synchronized panels.

In [ ]:
display_animation(
    result.animate(
        frames=None,
        interval=120,
    )
)

## Interpretation

All three runs use exactly the same potential, 16-vertex circle, initial condition, and observation times. The internal copies are re-embedded on the diagonal after every direct or adjoint map, so their separation after every internal stage should be zero up to round-off. The projection is non-invertible, therefore the projected symplectic defect is a diagnostic to evaluate rather than a quantity this method is expected to preserve. Differences between curves isolate the step-size behavior of this stage-projected formulation.